# Qwen3-0.6B × 커스텀 Triton AttentionBackend (paged KV inside kernel)

vLLM 의 공식 `AttentionBackend` 스펙을 서브클래싱하고,
**Python entry point (`vllm.general_plugins`) 로 자동 등록** 하는 정식 플러그인 경로.

**이 프로젝트의 학습 포인트**: split 과 동일한 커널 2개 구조를 유지하되,
두 커널 모두 **`block_table` 로 paged KV 를 커널 내부에서 직접 읽는다** (vLLM v0 스타일).
Python 쪽의 `_gather_kv` 는 완전히 제거. 어떤 작은 변화가 vLLM 추론의 메모리 효율성을
가능하게 하는지 체감하는 단계.

다음 단계 (max\_num\_seqs>1 + 혼합 배치 unified 커널): 상위 `vllm_unified` 프로젝트 (미래).


## 준비물 & 설치

- CUDA GPU 필수
- Python >= 3.10
- vLLM **0.19.1 정확히** 권장 (내부 API drift)

```bash
# 노트북 디렉토리에서
pip install -e .
```

`pyproject.toml` 의 entry point (`vllm.general_plugins = triton_attention_backend:register`) 가
모든 vLLM 프로세스(메인·엔진 코어·워커) 에서 자동으로 `register()` 를 호출해 준다.
즉 **노트북 안에서 `register()` 를 수동 호출하지 않는다**.

절차:
1. cell 3 — 환경 확인
2. cell 8 — 커널 smoke test
3. cell 10 — LLM 생성 (plugin 이 이미 CUSTOM 슬롯 점유)
4. cell 11~13 — generate + 실행 증거 확인


In [ ]:
import torch, triton, vllm

print('cuda:', torch.cuda.is_available())
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('vllm:', vllm.__version__)
print('triton:', triton.__version__)

if not torch.cuda.is_available():
    raise SystemExit('이 노트북은 CUDA GPU가 필요합니다.')

# 이 노트북은 vLLM 0.19.x 기준으로 작성됨. 다른 버전에서는 내부 API drift 가능.
assert vllm.__version__.startswith('0.19'), (
    f'vLLM 0.19.x 권장 (현재: {vllm.__version__}). '
    '다른 버전은 AttentionBackend 내부 API 가 다를 수 있음.'
)


## Qwen3-0.6B 구조 요약

| 항목 | 값 |
|---|---|
| hidden_size | 1024 |
| Q heads | 16 |
| KV heads | 8 (GQA 2:1) |
| head_dim | 128 |
| layers | 28 |

In [ ]:
from transformers import AutoConfig

cfg = AutoConfig.from_pretrained('Qwen/Qwen3-0.6B')
hd = cfg.head_dim if hasattr(cfg, 'head_dim') else cfg.hidden_size // cfg.num_attention_heads
print('hidden:', cfg.hidden_size)
print('Q heads:', cfg.num_attention_heads, '/ KV heads:', cfg.num_key_value_heads)
print('head_dim:', hd)
print('layers:', cfg.num_hidden_layers)

## Triton 커널 역할

`triton_attn.py` 에는 두 개의 `@triton.jit` 커널:

- **`_fwd_kernel_prefill_paged`** — q\_len == kv\_len (causal). paged read.
- **`_fwd_kernel_decode_paged`** — q\_len == 1, no causal mask. paged read.

두 커널 모두 다음 인자로 paged KV 를 직접 인덱싱:
- `K_cache, V_cache`   shape `[num_blocks, block_size, num_kv_heads, head_dim]`
- `block_table[seq, logical_block]` = 물리 block index
- `BLOCK_SIZE` (constexpr) = paged block size

```python
# 커널 내부에서 key position k 에 대한 로드
logical   = k // BLOCK_SIZE
slot      = k %  BLOCK_SIZE
phys      = block_table[seq_idx, logical]
k_vec     = K_cache[phys, slot, kv_head_idx, :]
```


In [ ]:
from triton_attn import (
    triton_attention_prefill_paged,
    triton_attention_decode_paged,
    _pack_to_paged,
)
print('prefill wrapper:'); print(triton_attention_prefill_paged.__doc__)
print()
print('decode  wrapper:'); print(triton_attention_decode_paged.__doc__)


In [ ]:
# 커널 smoke test (paged):
#   1) dense Q/K/V 를 랜덤 생성
#   2) SDPA 로 reference 계산
#   3) _pack_to_paged 로 paged cache 구성
#   4) 우리 paged 커널 호출 후 SDPA 와 비교
import torch, triton
import torch.nn.functional as F

B, Hq, Hkv, S, D = 1, 16, 8, 128, 128
q = torch.randn(B, Hq, S, D, dtype=torch.float16, device='cuda')
k = torch.randn(B, Hkv, S, D, dtype=torch.float16, device='cuda')
v = torch.randn(B, Hkv, S, D, dtype=torch.float16, device='cuda')

# reference (dense)
k_rep = k.repeat_interleave(Hq // Hkv, dim=1)
v_rep = v.repeat_interleave(Hq // Hkv, dim=1)
ref = F.scaled_dot_product_attention(q, k_rep, v_rep, is_causal=True)

# paged path
key_cache, value_cache, block_table = _pack_to_paged(k, v, block_size=16)
seq_lens = torch.full((B,), S, dtype=torch.int32, device='cuda')
ours = triton_attention_prefill_paged(q, key_cache, value_cache, block_table, seq_lens)

err = (ours.float() - ref.float()).abs().max().item()
print(f'prefill paged max_abs_err = {err:.6f}  →  {"PASS" if err < 1e-2 else "FAIL"}')

# decode (q=1) smoke
q_last = q[:, :, -1:, :].contiguous()
ref_last = ref[:, :, -1:, :]
ours_dec = triton_attention_decode_paged(q_last, key_cache, value_cache, block_table, seq_lens)
err_dec = (ours_dec.float() - ref_last.float()).abs().max().item()
print(f'decode  paged max_abs_err = {err_dec:.6f}  →  {"PASS" if err_dec < 1e-2 else "FAIL"}')


## Plugin entry point + paged KV read in-kernel

`pyproject.toml`:
```toml
[project.entry-points."vllm.general_plugins"]
my_paged_backend = "triton_attention_backend:register"
```

`pip install -e .` 이후 vLLM 이 모든 프로세스에서 `register()` 자동 호출.

**split 과의 차이**: backend `MyTritonImpl.forward` 에서 decode 시 `_gather_kv` 로 paged KV 를
Python 에서 모으던 코드가 **사라졌다**. 대신 커널이 `key_cache, value_cache, block_table, seq_lens`
를 받아 내부에서 간접 주소를 계산해 직접 로드.

```python
if q_len == s_len:   # prefill
    o = triton_attention_prefill_paged(q, key_cache, value_cache, block_table, seq_lens, scale)
else:                # decode
    o = triton_attention_decode_paged(q, key_cache, value_cache, block_table, seq_lens, scale)
```

이게 vLLM v0 의 실제 구조 — prefill 과 decode 가 별도 커널이지만 둘 다 paged cache 를 그대로 접근.

실행 증거는 엔진 코어 stderr 의 `fired (prefill paged) / (decode paged)` 로그로 확인.


In [ ]:
# entry point 등록 상태 확인 (pip install -e . 이 성공했다면 등록되어 있어야 함)
from importlib.metadata import entry_points

eps = list(entry_points(group='vllm.general_plugins'))
mine = [e for e in eps if e.name == 'my_paged_backend']
assert mine, (
    '`my_paged_backend` entry point 가 보이지 않는다. '
    '노트북 디렉토리에서 `pip install -e .` 을 실행했는지 확인하라.'
)
print('plugin registered:', mine[0].name, '->', mine[0].value)


In [ ]:
from vllm import LLM, SamplingParams
from vllm.v1.attention.backends.registry import AttentionBackendEnum

llm = LLM(
    model='Qwen/Qwen3-0.6B',
    dtype='float16',
    attention_backend=AttentionBackendEnum.CUSTOM,
    enforce_eager=True,
    max_num_seqs=1,
    max_model_len=2048,
)


In [ ]:
out = llm.generate(
    ['The capital of France is'],
    SamplingParams(temperature=0, max_tokens=32)
)
print(out[0].outputs[0].text)


## 검증: "실제로 우리 백엔드가 실행됐는가"

plugin 방식에서는 엔진 코어가 **별도 프로세스**에서 돌기 때문에
메모리 내 `FIRE_COUNTER` 는 이 노트북 프로세스에 전달되지 않는다 (읽으면 0).

대신 위 `llm.generate(...)` 출력 **바로 위** 에 엔진 코어 stderr 가 다음과 같이 찍혔어야 한다:

```
(EngineCore pid=XXXXX) INFO ... [cuda.py] Using AttentionBackendEnum.CUSTOM backend.
(EngineCore pid=XXXXX) WARNING ... MyTritonImpl.forward fired (prefill) tokens=5
(EngineCore pid=XXXXX) WARNING ... MyTritonImpl.forward fired (decode) s_len=6
```

두 `fired` 로그가 모두 찍혔으면 prefill/decode 양쪽에서 Triton 커널이 실제로 호출된 증거다.
(로그는 각 경로 첫 1 회만 기록 — `MY_TRITON_BACKEND_LOG=0` 으로 끌 수도 있음)


In [ ]:
# 이 노트북 프로세스에서 확인 가능한 것:
# (1) plugin 이 등록돼 있고
# (2) CUSTOM 슬롯에 우리 백엔드 경로가 꽂혔는가
from vllm.v1.attention.backends.registry import AttentionBackendEnum

path = AttentionBackendEnum.CUSTOM.get_path()
print('CUSTOM slot ->', path)
assert 'MyTritonBackend' in path, 'CUSTOM slot 에 우리 백엔드가 등록되지 않음'
print('OK — 노트북 프로세스에서도 plugin 자동 등록 확인')
print()
print('실제 커널 실행 증거는 위 cell 11 출력 바로 위의 `MyTritonImpl.forward fired ...` 로그를 확인할 것.')


## 다음 단계 — `vllm_unified`

- **`query_start_loc / seq_lens` 메타로 multi-seq 처리** — max\_num\_seqs > 1
- **한 커널이 prefill+decode 혼합 배치 처리** (continuous batching 의 진수)
- **split-k 최적화** — 긴 컨텍스트에서 parallelism 확보 (stage1/stage2 패턴)
- **fp8 KV cache** — 메모리 추가 절약
